# Document-level fact Knowledge Indicators + an agent harness

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kderusso/elasticsearch-labs/blob/kderusso/context-engine-notebooks/notebooks/context-engine/manual-walkthrough/index-facts-kis.ipynb)

The **Context Engine** gives AI agents structured, pre-computed **Knowledge Indicators (KIs)**, facts they can retrieve in a single call instead of scanning thousands of documents. This interactive notebook will walk you through how to set up a context engine, using the official [Elasticsearch Python client](https://www.elastic.co/guide/en/elasticsearch/client/python-api/current/index.html) and the [Kibana Workflows API](https://www.elastic.co/guide/en/kibana/current/index.html).

For this walkthrough, we use a sample from the [BrowseComp-Plus](https://github.com/texttron/BrowseComp-Plus) corpus.


## Create Elasticsearch environment

This notebook is designed to be run on an Elastic Serverless project. This functionality will be available in a future Elastic cloud release. If you don't have an Elastic Serverless project, sign up [here](https://cloud.elastic.co/registration?onboarding_token=search&cta=cloud-registration&tech=trial&plcmt=article%20content&pg=search-labs).

Once logged in to your Elastic Cloud account, go to the Create page and select Create project.


## Install packages and import modules

To get started, we'll need to connect to our Elastic deployment using the Python client. We connect using the Elasticsearch and Kibana endpoint URLs and an API key, which works for both Cloud deployments and Serverless projects.

In [ ]:
!pip install -q "elasticsearch>=9,<10" datasets requests pyyaml langchain-openai langgraph deepagents

### Initialize the Elasticsearch client

Instantiate the Elasticsearch Python client with your **Elasticsearch** and **Kibana** endpoint URLs and an API key — the same inputs work for both an Elastic Cloud deployment and a Serverless project (both expose direct endpoint URLs on their deployment/project page).

- [Create an API key](https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key)

In [ ]:
import json
import time
import requests
from getpass import getpass
from elasticsearch import Elasticsearch, helpers

ES_URL = input("Elasticsearch endpoint URL: ").strip().rstrip("/")
KIBANA_URL = input("Kibana endpoint URL: ").strip().rstrip("/")
ELASTIC_API_KEY = getpass("Elastic API key: ")

client = Elasticsearch(hosts=[ES_URL], api_key=ELASTIC_API_KEY)
print(client.info())

Next, we ensure that Kibana is accessible.

In [ ]:
def kbn_request(method, path, *, body=None, internal=False, api_version=None):
    """Call a Kibana REST API and return the parsed JSON response."""
    headers = {
        "Authorization": f"ApiKey {ELASTIC_API_KEY}",
        "kbn-xsrf": "true",
        "Content-Type": "application/json",
    }
    if internal:
        headers["x-elastic-internal-origin"] = "Kibana"
    if api_version:
        headers["elastic-api-version"] = api_version
    resp = requests.request(
        method,
        f"{KIBANA_URL}{path}",
        headers=headers,
        data=json.dumps(body) if body is not None else None,
    )
    resp.raise_for_status()
    return resp.json() if resp.text else {}

status = kbn_request("GET", "/api/status")
print("Kibana status:", status.get("status", {}).get("overall", {}).get("level"))

### Enable the Context Engine feature flag

The Context Engine is gated behind the `contextEngine:enabled` advanced setting, and is disabled by default.

> If the cell prints "Could not set feature flag", the setting may already be on or is managed at the deployment level. Check `Stack Management` → `Advanced Settings` and search for `Context Engine` to confirm it's enabled before continuing.

In [ ]:
try:
    kbn_request("POST", "/internal/kibana/settings",
                body={"changes": {"contextEngine:enabled": True}}, internal=True)
    print("Enabled feature flag: contextEngine:enabled")
except Exception as e:
    print(f"Could not set feature flag via API: {e}")
    print("Verify that the `contextEngine:enabled` feature flag is toggled ON in Stack Management → Advanced Settings.")

### Set up the deep agent harness

Both answers in this notebook come from the same [LangChain deep agents](https://pypi.org/project/deepagents/) harness: same model, same system prompt, same question. The only thing we swap between the baseline and the Context Engine is the search tool the agent gets. Each `ask_agent` call also prints the tokens the agent spent (summed over every model call in the run), so you can compare the inference cost of the two paths. Note the KI path has an indexing-time cost this doesn't capture: the workflow's `ai.agent` step already spent tokens generating each KI. The harness talks to any OpenAI-compatible endpoint — press Enter at the prompts to accept the defaults (OpenRouter + Claude Haiku), or point it at your own provider and model.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.callbacks import get_usage_metadata_callback
from deepagents import create_deep_agent

# Any OpenAI-compatible endpoint works (OpenRouter, OpenAI, a local vLLM/Ollama, ...).
# Press Enter at the prompts to accept the defaults: OpenRouter + Claude Haiku.
LLM_BASE_URL = input("LLM base URL [https://openrouter.ai/api/v1]: ").strip() or "https://openrouter.ai/api/v1"
LLM_MODEL = input("LLM model [anthropic/claude-haiku-4-5]: ").strip() or "anthropic/claude-haiku-4-5"
LLM_API_KEY = getpass("LLM API key: ")

# The deep agent plans and may chain several tool calls, so give it output headroom.
agent_llm = ChatOpenAI(
    model=LLM_MODEL,
    api_key=LLM_API_KEY,
    base_url=LLM_BASE_URL,
    max_tokens=4096,
)

SYSTEM_PROMPT = """You are a research assistant answering questions about a document corpus.
You have NOT memorized the corpus. Whenever a question depends on specific facts, names, dates,
organizations, or events, call your search tool to retrieve relevant material before answering.
Ground your answer strictly in what the tool returns, and cite the titles of the sources you
used. If the retrieved context does not contain the answer, say so plainly rather than guessing."""

def ask_agent(question, search_tool):
    """Build a deep agent around a single search tool and ask it a question."""
    agent = create_deep_agent(model=agent_llm, tools=[search_tool], system_prompt=SYSTEM_PROMPT)
    with get_usage_metadata_callback() as cb:
        result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    usage = next(iter(cb.usage_metadata.values()), {})
    print(f"[tokens] input={usage.get('input_tokens', 0):,}  "
          f"output={usage.get('output_tokens', 0):,}  total={usage.get('total_tokens', 0):,}\n")
    return result["messages"][-1].content

## Index a sample from the BrowseComp-Plus dataset

The [BrowseComp-Plus corpus](https://huggingface.co/datasets/Tevatron/browsecomp-plus-corpus) is ~100k web documents (~1.76 GB); each row has `docid`, `text`, and `url`. We stream the dataset and ingest the first `SAMPLE_DOCS` rows into an Elasticsearch index, deriving a a lightweight title from the first line of each document and attacing mapping metadata so the index is self-describing.

The cell is idempotent: if the index already exists with documents, it's reused and indexing is skipped — so you can reuse the same index without re-running setup.

In [ ]:
INDEX_NAME = "browsecomp-plus"
SAMPLE_DOCS = 50  # documents to index (we generate one KI per indexed doc, so it's best to keep this relatively small)

if client.indices.exists(index=INDEX_NAME) and client.count(index=INDEX_NAME)["count"] > 0:
    count = client.count(index=INDEX_NAME)["count"]
    print(f"Index '{INDEX_NAME}' already has {count} documents — reusing it (skipping indexing).")
else:
    client.indices.delete(index=INDEX_NAME, ignore_unavailable=True)
    client.indices.create(
        index=INDEX_NAME,
        mappings={
            "_meta": {
                "description": (
                    "BrowseComp-Plus corpus: ~100k human-verified web documents "
                    "(news articles, Wikipedia entries, institutional pages) used as a "
                    "reasoning-intensive browsing/QA retrieval benchmark. BM25-only index."
                )
            },
            "properties": {
                "docid": {"type": "keyword", "meta": {"description": "Stable corpus document id."}},
                "url": {"type": "keyword", "meta": {"description": "Source URL the document was crawled from."}},
                "title": {"type": "text", "meta": {"description": "Document title (from the document's front matter)."}},
                "text": {"type": "text", "meta": {"description": "Full document text: title, date, and body content."}},
            },
        },
    )

    import re
    from datasets import load_dataset

    corpus = load_dataset("Tevatron/browsecomp-plus-corpus", split="train", streaming=True)

    def extract_title(text):
        # Each BrowseComp-Plus doc opens with a YAML front-matter block (--- ... ---)
        # whose `title:` field holds the real title; the first body line is just the
        # "---" delimiter. Read the title from there, else fall back to the first real line.
        m = re.search(r"^title:\s*(.+)$", text, flags=re.MULTILINE)
        if m:
            return m.group(1).strip()[:200]
        for line in text.splitlines():
            s = line.strip()
            if s and s != "---":
                return s[:200]
        return ""

    def actions(n):
        for i, row in enumerate(corpus):
            if i >= n:
                break
            text = row["text"]
            title = extract_title(text)
            yield {
                "_index": INDEX_NAME,
                "_id": row["docid"],
                "_source": {"docid": row["docid"], "url": row["url"], "title": title, "text": text},
            }

    helpers.bulk(client, actions(SAMPLE_DOCS))
    client.indices.refresh(index=INDEX_NAME)
    print(f"Indexed {client.count(index=INDEX_NAME)['count']} documents into '{INDEX_NAME}'.")

## Baseline: Answer the question with the deep agent, without using KIs. 

Before creating any KIs, establish the baseline: the agent's only tool is a plain BM25 search over the raw index, returning document chunks. Later, we will swap this tool for the Context Engine and ask the same question again.

In [ ]:
QUESTION = "What was the actress who played Torvi from Vikings also known for?"

@tool
def search_documents(query: str) -> str:
    """Search the raw document corpus (BM25 lexical match over title and text).

    Call this whenever answering depends on specific facts, names, dates, organizations, or
    events that would live in the corpus rather than in your own training data. Pass a focused
    natural-language query. Returns the top matching documents as title + text excerpt.
    """
    resp = client.search(
        index=INDEX_NAME,
        body={
            "query": {"multi_match": {"query": query, "fields": ["title", "text"]}},
            "size": 5,
            "_source": ["title", "text"],
        },
    )
    hits = resp["hits"]["hits"]
    if not hits:
        return "No matching documents found."
    return "\n\n".join(
        f"[{h['_source'].get('title', '')}]\n{h['_source'].get('text', '')[:600]}"
        for h in hits
    )

print("=== Agent answer from BM25 search (raw document chunks) ===\n")
print(ask_agent(QUESTION, search_documents))

## Create a Workflow to generate KIs

Given a `docId`, the Workflow calls a `foreach` (a one-row loop) runs two steps:

| Step | Type | What it does |
|------|------|--------------|
| `generate_ki` | `ai.agent` | Read the document and extract a retrieval-optimized Knowledge Indicator as **structured output** (title, summary, questions it answers, key entities, topics, tagline). |
| `sink_ki` | `contextEngine.addEntry` | Upsert the generated KI into the Context Engine, namespaced under the `corpus_entry` type. |

We create the workflow once. To balance speed and simplicity, this workflow runs once per document, looking up the document by `docid` using ES|QL. Because each execution is self-contained, we execute them in parallel from the client (next section) so the per-doc `ai.agent` calls run concurrently instead of one at a time. In practice, we'd likely want to execute `workflow.executeAsync` or call native parallelization in the Workflow.

### Point the workflow at your GenAI connector (optional)

Leave blank to use Kibana's default GenAI connector, or paste a specific connector id to pin one.

In [ ]:
LLM_CONNECTOR_ID = input("Kibana GenAI connector id (blank = default connector): ").strip()

### Define the workflow

In [ ]:
_WORKFLOW_YAML_TEMPLATE = """
version: '1'
name: browsecomp-plus-per-doc-ki
description: Look up a single BrowseComp-Plus document by id with ES|QL, generate a KI with an AI agent, and sink it into the Context Engine as a corpus_entry.
enabled: true
tags:
  - context-engine
  - browsecomp-plus
triggers:
  - type: manual
steps:
  - name: query_corpus
    type: elasticsearch.esql.query
    with:
      # One workflow execution handles a single document (passed in as inputs.docid),
      # so the client can fan many executions out in parallel.
      # Column order drives the foreach.item[N] indices below:
      #   item[0]=docid  item[1]=title  item[2]=url  item[3]=text
      # SUBSTRING keeps the prompt bounded.
      query: >
        FROM __INDEX_NAME__
        | WHERE docid == "{{ inputs.docid }}"
        | KEEP docid, title, url, text
        | EVAL text = SUBSTRING(text, 1, 12000)
        | LIMIT 1

  - name: loop_corpus_docs
    type: foreach
    foreach: '{{steps.query_corpus.output.values}}'
    steps:
      # Turn the raw doc into a retrieval-optimized Knowledge Indicator (structured output).
      - name: generate_ki
        type: ai.agent
        connector-id: "__LLM_CONNECTOR_ID__"
        timeout: 120s
        with:
          message: >
            You are a knowledge engineer building a Knowledge Indicator (KI) for an
            enterprise document-retrieval corpus. A KI is a compact, high-signal record
            that a hybrid (BM25 + semantic) search engine and an AI agent use to FIND and
            JUDGE the source document without reading it in full.

            Read the document below and extract a faithful, richly structured KI. Be 100%
            grounded: never state anything not supported by the text. Prefer concrete,
            named specifics (people, organizations, products, dates, places, figures) over
            vague phrasing. If a field cannot be determined from the text, return an empty
            string or empty array rather than guessing.

            Document ID: {{ foreach.item[0] }}
            Original Title: {{ foreach.item[1] }}
            Source URL: {{ foreach.item[2] }}
            Document Body:
            {{ foreach.item[3] }}
          schema:
            type: object
            properties:
              title:
                type: string
                description: A concise, specific, human-readable title (<= 12 words).
              summary:
                type: string
                description: A dense 3-5 sentence factual summary capturing the document's main claims, named entities, and conclusions. The PRIMARY semantic search surface.
              answers_questions:
                type: array
                items:
                  type: string
                description: 2-5 natural-language questions this document can authoritatively answer.
              key_entities:
                type: array
                items:
                  type: string
                description: 3-10 salient named entities (people, organizations, products, places, dates) explicitly mentioned.
              topics:
                type: array
                items:
                  type: string
                description: 3-8 short topic/category labels describing what the document is about.
              tagline:
                type: string
                description: A single ultra-short phrase (<= 6 words) capturing the essence of the document.
            required:
              - title
              - summary
              - answers_questions
              - key_entities
              - topics

      # Sink the generated KI into the Context Engine.
      - name: sink_ki
        type: contextEngine.addEntry
        with:
          originId: '{{ foreach.item[0] }}'
          attachmentType: corpus_entry
          action: upsert
          chunks:
            - type: corpus_entry
              title: '{{ foreach.item[1] | default: steps.generate_ki.output.structured_output.title }}'
              content: >
                === SOURCE / PROVENANCE ===
                Backing Elasticsearch index: __INDEX_NAME__
                Document ID (docid): {{ foreach.item[0] }}
                Source URL: {{ foreach.item[2] }}
                Retrieve the full original document with ES|QL:
                FROM __INDEX_NAME__ | WHERE docid == "{{ foreach.item[0] }}"
                === KNOWLEDGE INDICATOR ===
                {{ steps.generate_ki.output.structured_output.summary }}
                Questions this document answers: {{ steps.generate_ki.output.structured_output.answers_questions | join: " | " }}
                Key entities: {{ steps.generate_ki.output.structured_output.key_entities | join: ", " }}
              description: >
                {{ steps.generate_ki.output.structured_output.tagline }}.
                Topics: {{ steps.generate_ki.output.structured_output.topics | join: ", " }}.
                Entities: {{ steps.generate_ki.output.structured_output.key_entities | join: ", " }}.
"""

# Point the workflow at our index (single source of truth = INDEX_NAME).
_yaml = _WORKFLOW_YAML_TEMPLATE.replace("__INDEX_NAME__", INDEX_NAME)

# Pin a connector if one was supplied; otherwise drop the line so ai.agent uses
# Kibana's default GenAI connector (step-level config can't be templated).
if LLM_CONNECTOR_ID:
    WORKFLOW_YAML = _yaml.replace("__LLM_CONNECTOR_ID__", LLM_CONNECTOR_ID)
else:
    WORKFLOW_YAML = "\n".join(
        line for line in _yaml.splitlines() if "__LLM_CONNECTOR_ID__" not in line
    )

print(WORKFLOW_YAML)

### Create and run the workflow

We create the workflow once, then (in the next cell) run one execution per document, up to `MAX_PARALLEL_KIS` at a time. Each execution makes one `ai.agent` call. Running them in parallel makes this process go faster, but it still will take a few minutes to complete.

In [ ]:
WF_API_VERSION = "2023-10-31"
TERMINAL_STATES = {"completed", "failed", "cancelled", "timed_out", "skipped"}

def create_workflow(yaml_str):
    return kbn_request("POST", "/api/workflows/workflow",
                       body={"yaml": yaml_str}, api_version=WF_API_VERSION)

def run_workflow(workflow_id, inputs):
    return kbn_request("POST", f"/api/workflows/workflow/{workflow_id}/run",
                       body={"inputs": inputs}, api_version=WF_API_VERSION)

def wait_for_execution(execution_id, timeout=600, interval=5):
    deadline = time.time() + timeout
    while time.time() < deadline:
        ex = kbn_request("GET", f"/api/workflows/executions/{execution_id}?includeOutput=true",
                         api_version=WF_API_VERSION)
        if ex["status"] in TERMINAL_STATES:
            return ex
        time.sleep(interval)
    raise TimeoutError(f"Execution {execution_id} did not finish within {timeout}s")

wf = create_workflow(WORKFLOW_YAML)
workflow_id = wf["id"]
print("Created workflow:", workflow_id)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_PARALLEL_KIS = 8   # how many workflow executions to run at once
PROGRESS_EVERY = 10    # print a running count every N completed KIs
MAX_ATTEMPTS = 3       # per-doc attempts before treating a failure as real
RETRY_BACKOFF = 5      # seconds between attempts, multiplied by the attempt number

# One execution per document — fan them out so the LLM calls run concurrently.
docids = [
    hit["_id"]
    for hit in helpers.scan(client, index=INDEX_NAME, _source=False, query={"query": {"match_all": {}}})
]
print(f"Generating one KI per document for {len(docids)} docs ({MAX_PARALLEL_KIS} in parallel)...\n")

def generate_ki_for(docid):
    """Run the workflow for one doc, retrying transient failures up to MAX_ATTEMPTS.

    The ai.agent inference call fails intermittently (e.g. a transient empty-content
    400), so a single failure isn't necessarily a real one — retry before giving up.
    """
    last = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            execution = run_workflow(workflow_id, {"docid": docid})
            result = wait_for_execution(execution["workflowExecutionId"])
            if result["status"] == "completed":
                return docid, result
            last = result
        except Exception as e:
            last = {"status": "error", "error": {"message": str(e)}}
        if attempt < MAX_ATTEMPTS:
            print(f"  retry {attempt}/{MAX_ATTEMPTS - 1} for docid={docid} (was: {last['status']})")
            time.sleep(RETRY_BACKOFF * attempt)
    return docid, last  # retries exhausted — caller treats this as a real failure

completed = 0
failure = None
with ThreadPoolExecutor(max_workers=MAX_PARALLEL_KIS) as pool:
    futures = {pool.submit(generate_ki_for, d): d for d in docids}
    for future in as_completed(futures):
        docid, result = future.result()
        if result["status"] != "completed":
            # Abort only after retries are exhausted: cancel anything not yet started so
            # we stop creating KIs until the problem is fixed. (In-flight runs still finish.)
            failure = (docid, result)
            for f in futures:
                f.cancel()
            break
        completed += 1
        if completed % PROGRESS_EVERY == 0 or completed == len(docids):
            print(f"  {completed}/{len(docids)} KIs completed")

if failure:
    docid, result = failure
    print(f"\nAborted — docid={docid} still failing after {MAX_ATTEMPTS} attempts (status: {result['status']}).")
    if result.get("error"):
        print("Error:", result["error"].get("message", result["error"]))
    for step in result.get("stepExecutions", []):
        if step.get("status") == "failed":
            print("Failed step:", step.get("stepId"), "->", json.dumps(step.get("error")))
    raise RuntimeError(f"Aborted KI generation: docid={docid} failed after {MAX_ATTEMPTS} attempts; fix the issue and re-run.")

print(f"\nDone: {completed}/{len(docids)} KIs completed.")

## View an example KI

We'll use the GET API to view the structure of a KI. For this example we'll take the first matching KI for our question. 

TODO: Update this to use renamed API 

## Answer the question from KIs with a deep agent

Now the Context Engine side of the comparison: the same agent harness, system prompt, and question — only the search tool changes. Instead of BM25 over raw documents, the agent retrieves from the Context Engine, where each result is a pre-computed Knowledge Indicator: structured, summarized, and semantically indexed by the workflow.

First, a thin wrapper over the Context Engine's `_search` endpoint.

TODO: Update this when the `get_context` call is available.

In [ ]:
def get_context(query, size=5, types=None):
    body = {
        "query": query,
        "size": size,
        "fields": ["content", "description", "tags", "references"],
    }
    if types:
        body["filters"] = {"types": types}
    return kbn_request("POST", "/internal/agent_context_layer/sml/_search",
                       body=body, internal=True)

### View an example KI

We'll use the GET API to view the structure of a KI. For this example we'll take the first matching KI for our question. 

TODO: Update this to use renamed API 

In [ ]:
# Pick one KI (the top search hit) and fetch its full record by (type, originId).
# The search hit's origin.uri is "type://originId" — split it to address the KI.
hits = get_context(QUESTION, size=1, types=["corpus_entry"])["results"]
if not hits:
    print("No KIs found — run the workflow cells above first.")
else:
    ki_type, origin_id = hits[0]["origin"]["uri"].split("://", 1)
    print(f"GET /internal/agent_context_layer/sml/{ki_type}/{origin_id}\n")
    ki = kbn_request("GET", f"/internal/agent_context_layer/sml/{ki_type}/{origin_id}", internal=True)
    print(json.dumps(ki, indent=2))

### Wrap `get_context` as a tool for the deep agent

In [ ]:
@tool
def search_context(query: str) -> str:
    """Search the Context Engine for Knowledge Indicators (KIs) extracted from the document corpus.

    Call this whenever answering depends on specific facts, names, dates, organizations, or
    events that would live in the corpus rather than in your own training data. Pass a focused
    natural-language query. Returns the most relevant KIs with their title and content
    (each KI includes provenance so you can cite the source document).
    """
    results = get_context(query, size=5, types=["corpus_entry"])["results"]
    if not results:
        return "No relevant context found in the Context Engine."
    return "\n\n".join(f"[{r['title']}]\n{r.get('content', '')}" for r in results)

Hand the tool to the same agent harness and ask the same question. The agent decides on its own to call `search_context`, reads the returned KIs, and answers from them — compare with the baseline answer from before.

In [ ]:
print("=== Agent answer from the Context Engine (KIs) ===\n")
print(ask_agent(QUESTION, search_context))

That is the full Context Engine loop: documents went in, the workflow distilled each into a fact KI, `get_context` made them retrievable, and an external agent answered a question by calling `get_context` as a tool and grounding its response in the returned KIs.

## Clean up

Remove the KIs from the Context Engine, delete the workflow, and drop the Elasticsearch index. The feature flag remains enabled unless you explicitly disable it.

TODO Update CRUD API calls to renamed API 

In [ ]:
seen = set()
for hit in get_context("knowledge indicator document", size=100, types=["corpus_entry"])["results"]:
    uri = hit["origin"]["uri"]
    if uri in seen:
        continue
    seen.add(uri)
    ki_type, _, origin_id = uri.partition("://")
    kbn_request("DELETE", f"/internal/agent_context_layer/sml/{ki_type}/{origin_id}", internal=True)
print("Deleted KIs")

try:
    kbn_request("DELETE", f"/api/workflows/workflow/{workflow_id}", api_version=WF_API_VERSION)
    print("Deleted workflow:", workflow_id)
except Exception as e:
    print("Workflow delete skipped:", e)

client.indices.delete(index=INDEX_NAME, ignore_unavailable=True)
print("Deleted index:", INDEX_NAME)